# Hypothesis Testing
## Single Proportion Test

## Jobs Training Programs

- Suppose the national unemployment rate is **30%**
- A jobs program trains 60 people; **15 are unemployed** (25%)
- The organization claims this proves the program works
- Is that difference (25% vs 30%) meaningful or just random?

## Create the Data

Now let's create some data to match our hypothetical example.

In [ ]:
library(tidyverse)

jobs_program <- tibble(
  outcome = c(rep("unemployed", 15), rep("employed", 45))
)

jobs_program |>
  ggplot(aes(x = outcome)) +
  geom_bar(fill = "chartreuse4") + theme_bw()

## Question

Is it possible to assess this hypothetical organization's claim using the data and information presented thus far?

> *"Our jobs program is a **success** because only 15 of the 60 people that we trained did not have a job. Thus our 25% unemployment rate beats the country's unemployment rate of 30%."*

## Correlation vs. Causation

- **No.** We need to know more about how people were selected for the program in order to assess causality (e.g., were they randomly assigned?)
- But, we can still ask whether the unemployment rate of $\hat{p}$ = 0.25 could be due to chance.
- That's a question about the **sampling variability**.
- Now we'll use a **hypothesis test** to answer it.

## Hypothesis Testing Intuition

1. We are going to assume **"nothing is going on"**
   - In this case, the jobs program had no impact

2. We figure out what the **distribution of outcomes** might look like if nothing is going on
   - Here: if we take a sample of 60 from a population where the proportion parameter is 0.30

3. We assess **how likely** we would be to observe our data if nothing is going on
   - If very unlikely, we conclude that something is probably going on

## Stating our Hypotheses

**Null hypothesis ($H_0$):** "There is nothing going on."
> Unemployment rate among those in the jobs program is no different than the country average of 30%.

**Alternative hypothesis ($H_A$):** "There is something going on."
> Unemployment rate is **lower** than the country average of 30%.

## Hypothesis Test

- **Hypothesis test:** If the null hypothesis were true, is the data we have in our sample likely to have been generated by chance (due to random variability)?
- If **yes** → we do **NOT** reject the null hypothesis
- If **not very likely** → we **reject** the null hypothesis

## Hypothesis Testing Framework

1. Start with the **null hypothesis** $H_0$, representing the status quo
2. Set an **alternative hypothesis** $H_A$, representing the research question
3. Conduct a hypothesis test under the assumption that the null hypothesis is true:
   - If the test results suggest the data do **not** provide convincing evidence for $H_A$ → stick with $H_0$
   - If they **do** → reject $H_0$ in favor of $H_A$

## p-values and Critical Values

- A **p-value** is the probability of the observed or more extreme outcome given that the **null hypothesis is true**
- A **critical value** ($\alpha$) is the threshold at which we will reject the null hypothesis
- If the p-value is less than $\alpha$, we **reject** the null hypothesis
- A standard threshold for $\alpha$ is **0.05**

## The Null Distribution

- Since $H_0: p = 0.30$, we need to simulate a null distribution where the probability of success (unemployment) for each trial (person in program) is 0.30

- We want to know how likely we would be to get an unemployment rate of 0.25 in our sample of 60, *if the true unemployment rate were 0.30*

$$P(\hat{p} \le 0.25 \mid p = 0.30)$$

We to simulate the null distribution many times...

We can use the `tidymodels` package to help with this process.

In [ ]:
# load tidymodels
library(tidymodels)

# simulate the distribution
null_dist <- jobs_program |>
  specify(response = outcome, success = "unemployed") |>
  hypothesize(null = "point", p = c("unemployed" = 0.30, "employed" = 0.70)) |>
  generate(reps = 2000, type = "draw") |> 
  calculate(stat = "prop")

What is being stored in *null_dist*?

In [ ]:
null_dist |>
  mutate(
    replicate = as.numeric(replicate),
    stat = round(stat, 3)
    )

Where should this distribution be centered? What should the mean be?

In [ ]:
null_dist |>
  summarize(mean = mean(stat))

Visualizing the Null Distribution

In [ ]:
ggplot(data = null_dist, mapping = aes(x = stat)) +
  geom_histogram(bins = 15, fill = "chartreuse4", color = "white") +
  labs(title = "Null distribution") + theme_bw()

Calculate the p-value

**p-value:** the proportion of simulated samples that are as extreme or more extreme than what we observed, assuming the null hypothesis is true.

In [ ]:
null_dist |>
  filter(stat <= (15/60)) |>
  summarize(p_value = n()/nrow(null_dist))

Visualizing the p-value

In [ ]:
ggplot(data = null_dist, mapping = aes(x = stat)) +
  geom_histogram(bins = 15, fill = "chartreuse4", color = "white") +
  labs(title = "Null distribution") +     
  geom_vline(xintercept = .25, linetype="dotted", 
                color = "black", size=1) + theme_bw()

p-value using `infer`

In [ ]:
library(infer)

# calculate the p-value from null distribution
p_value <- null_dist |>
  get_p_value(obs_stat = 15/60, direction = "less")

p_value

Visualizing the p-value using `infer`

In [ ]:
visualize(null_dist) +
  shade_p_value(obs_stat = 15/60, direction = "less")

## "Significance" Level

- Conventionally, people use a p-value of **0.05** as a cutoff ("significance level") for determining statistical significance
  - i.e., whether the null hypothesis should be rejected
  - i.e., whether the data is very unlikely to have been generated due to chance
- Always remember that this is a **convention**
  - *p = 0.049* is under the cutoff, while *p = 0.051* is not — are these really different?
- When people report "statistically significant" results, they mean the p-value is less than 0.05

## Our Hypothetical Study

- **Our finding:** if the true unemployment rate were 30% and we draw samples of 60, about 23% of the time we will get an unemployment rate lower than the one among program participants — simply due to random chance.
- **What should we conclude?**

## Conclusion

We do **NOT** reject the null hypothesis: the unemployment rate in the sample could likely have been **due to chance**.

## Try it out!

- What if the unemployment rate for the program was only **15%**?
  - Would you reject the null hypothesis in this case?
  - Demonstrate by calculating the p-value
- Try changing the true unemployment rate in the null distribution to **0.50 (50%)**
  - Simulate the null distribution
  - Would you reject the null hypothesis if the observed unemployment rate was 23% in this case?

In [ ]:
# Your code here